# Geospatial Data Analysis Lab: Steel Plants Dataset


**Learning Objectives:**
- Perform exploratory data analysis (EDA) on geospatial datasets
- Visualize geospatial data using interactive maps with Plotly
- Merge granular exposure / population data (LitPop) with asset locations
- Aggregate data at the company level
- Integrate geospatial visualizations into a Streamlit dashboard

**Dataset:** Download the Global Energy Monitor [Global Iron and Steel Tracker](https://globalenergymonitor.org/projects/global-iron-steel-tracker) (formerly Global Steel Plant Tracker). Use the plant-level download from that page and place the file in the same folder as this notebook (or update the load path accordingly).

---


## Submission info

Work in **groups of up to 4**. Fill in every member before submitting.

| # | Full name | Student ID |
|---|-----------|------------|
| 1 | Adrien Labergere | B00822082 |
| 2 | Louis Bettelli | B00820941 |
| 3 | Oscar Moulet | B00821262 |
| 4 | Juliette Guilloux-Pinard | B00825063 |
| 5 | Apolline Seguin | B00822617 |

**Group / repo name:** `aidams-lab1-moulet-bettelli-labergere-seguin-guilloux_pinard`  
**Submitter (one person):** Oscar Moulet (GitHub: racso-vs)  
**Repo URL:** https://github.com/racso-vs/aidams-lab1-moulet-bettelli-labergere-seguin-guilloux_pinard.git  
**Streamlit Cloud URL (bonus):** https://aidams-lab1-steel-plants.streamlit.app  

### What to submit
- This notebook (`lab_1.ipynb`) with all parts completed and cells run
- `app.py` (Part 6)
- Processed data exports used by the dashboard (e.g. CSV/Parquet), if applicable
- (Bonus) Deployed Streamlit Cloud app link, if completed

<span style="color: #FFD700; font-weight: bold">Send submission info to my email (1 email per group)</span> — include the GitHub repo URL and, if you did the bonus, the Streamlit Cloud link.


## Upload to GitHub

Follow this checklist (one repo per group):

1. Create a **private** repository (or use the course organization if provided).
2. Name it using the pattern above, e.g. `aidams-lab1-ali-ben-chen-diaz`.
3. Add your files (`lab_1.ipynb`, `app.py`, exports, and a short `README.md` with how to run the dashboard).
5. Commit and push:
   ```bash
   git init
   git add lab_1.ipynb app.py .gitignore README.md
   git commit -m "Complete AIDAMS Lab 1"
   git branch -M main
   git remote add origin <YOUR_REPO_URL>
   git push -u origin main
   ```
6. Invite the instructor (or open the assignment link) and paste the **repo URL** in the Submission info table above.

**Done when:** all 4 names are filled in, the notebook contains all outputs (no need for the instructor to re-run it), maps are visible, and `streamlit run app.py` works from the repo.

**Bonus (optional):** deploy `app.py` to Streamlit Cloud and paste the public app URL above / in your submission email.


## Part 1: Setup and Data Loading

Import the necessary libraries and load the steel plants dataset.

**Tip:** After loading, run `df.columns` and `df.head()`. Column names in the file may differ slightly by release — inspect them and adapt your code accordingly.


In [71]:
# Import required libraries
# - pandas for data manipulation
# - numpy for numerical operations
# - plotly.express and plotly.graph_objects for interactive visualizations
# - Any other libraries you need

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


In [72]:
# Load the steel plants dataset
# Tip: start with df.columns / df.head() and adapt names if your file differs slightly
#
# Common columns in this steel plant dataset:
# - Plant name (English)
# - Owner
# - Country/Area, Region
# - Coordinates  (often a single "lat, lon" string — not separate latitude/longitude columns)
# - Plant age (years)
# - Capacity field: usually "Nominal crude steel capacity (ttpa)" (check your df.columns to confirm the exact name)
#   (Older datasets may also include additional fields like ferronickel/sinter/coking/pelletizing capacities, 
#    but most analyses focus on nominal crude steel capacity as the main output.)

DATA_FILE = "Plant-level_data_Global_Iron_and_Steel_Tracker_June_2026_V1.xlsx"

# capacity isn't on the main "Plant data" sheet, it's in its own sheet
# so we load both
plants = pd.read_excel(DATA_FILE, sheet_name="Plant data")
capacities = pd.read_excel(DATA_FILE, sheet_name="Plant capacities and status")

CAP_COL = "Nominal crude steel capacity (ttpa)"

# After having a look at the excel, a few rows have ">0" instead of a real number, we turn those into NaN
capacities[CAP_COL] = pd.to_numeric(capacities[CAP_COL], errors="coerce")

# only keep units that are actually running, then sum per plant
# (some plants have more than one production unit)
operating_capacity = (
    capacities[capacities["Status"] == "operating"]
    .groupby("GEM plant ID")[CAP_COL]
    .sum()
    .rename("Capacity (ttpa)")
)

# bring capacity back into the main plant table
df = plants.merge(operating_capacity, on="GEM plant ID", how="left")

print(df.shape)
df.head()


(1293, 45)


,GEM plant ID,Plant name (English),Plant name (other language),Other plant names (English),Other plant names (other language),Owner,Owner (other language),Owner GEM entity ID,Owner PermID,SOE status,...,Steel sector end users,Workforce size,ISO 14001,ISO 50001,ResponsibleSteel certification,Main production equipment,Power source,Iron ore source,Met coal source,Capacity (ttpa)
0,P100000120882,Aba Iron and Steel Payas plant,ABA DEMİR ÇELİK,"EEY Iron and Steel, Nursan Steel Payas Plant (...",NaN,ABA Çelik Demir LŞ,ABA Demir ve Çelik İthalat İhracat Ticaret,E100000131190,unknown,NaN,...,unknown,900,unknown,unknown,no,EAF,unknown,unknown,NaN,1100.0
1,P100000120753,Abba Steel Ohangwena steel plant,NaN,Groot Suisse Oshana plant,NaN,Abba Steel Ltd,NaN,E100001012072,unknown,NaN,...,building and infrastructure,5500,unknown,unknown,no,EAF,unknown,unknown,NaN,NaN
2,P100000120802,Abinsk Electric Steel Works,АЭМЗ,"AESW, ASW, AEMZ, AEMK",Абинский ЭлектроМеталлургический завод,Abinski Elektrometallurgicheski Zavod LLC,"ООО ""АБИНСКИЙ ЭЛЕКТРОМЕТАЛЛУРГИЧЕСКИЙ ЗАВОД""",E100000130999,5039667129,NaN,...,unknown,4500,2025-10-06 00:00:00,unknown,no,EAF,unknown,NaN,unknown,1600.0
3,P100000120020,Abul Khair Steel Sitakunda plant,আবুল খায়ের স্টিল মেল্টিং লিমিটেড,"AKS Long Steel, AKS Sitakund, AKS Melting, Abu...",NaN,Abul Khair Steel Ltd,NaN,E100000131068,5074007077,NaN,...,"building and infrastructure, energy",unknown,unknown,unknown,no,EAF,"100MW Power Plant, in progress 50MW solar plant",unknown,unknown,1400.0
4,P100000120620,Acciaierie d'Italia Taranto steel plant,NaN,"ILVA Taranto steel plant (predecessor), ILVA S...",NaN,Acciaierie d'italia SpA,NaN,E100001010116,5067495106,Full,...,"automotive, building and infrastructure, energ...",11000,2025-04-30 00:00:00,2025-12-04 00:00:00,no,DRI; EAF; BF; BOF,unknown,unknown,unknown,NaN


---
## Part 2: Exploratory Data Analysis

Answer the following questions through your analysis:


### Question 1: Data Overview
**Task:** Display basic information about the dataset.
- How many steel plants are in the dataset?
- What are the column names and data types? (use `df.columns` / `df.dtypes` or `df.info()`, then adapt later code to the names you see)
- Are there any missing values?


In [73]:
# Display dataset shape

# rows = number of plants, columns = number of fields we have per plant
print(df.shape)

# so output gives us : 1293 plants

(1293, 45)


In [74]:
# Display column information and data types
# Start here: print(df.columns) and adapt column names in later cells if needed

print(df.columns)
df.dtypes


Index(['GEM plant ID', 'Plant name (English)', 'Plant name (other language)',
       'Other plant names (English)', 'Other plant names (other language)',
       'Owner', 'Owner (other language)', 'Owner GEM entity ID',
       'Owner PermID', 'SOE status', 'Parent (English)',
       'Parent GEM entity ID', 'Parent PermID', 'Location address',
       'Location address (other language)', 'Municipality', 'Subnational unit',
       'Country/area', 'Region', 'Coordinates', 'Coordinate accuracy',
       'GEM wiki page', 'Plant age', 'Announced date', 'Construction date',
       'Start date', 'Pre-retirement announcement date', 'Idled date',
       'Retired date', 'Ferronickel capacity (ttpa)',
       'Sinter plant capacity (ttpa)', 'Coking plant capacity (ttpa)',
       'Pelletizing plant capacity (ttpa)', 'Category steel product',
       'Steel products', 'Steel sector end users', 'Workforce size',
       'ISO 14001', 'ISO 50001', 'ResponsibleSteel certification',
       'Main production equ

GEM plant ID                           object
Plant name (English)                   object
Plant name (other language)            object
Other plant names (English)            object
Other plant names (other language)     object
Owner                                  object
Owner (other language)                 object
Owner GEM entity ID                    object
Owner PermID                           object
SOE status                             object
Parent (English)                       object
Parent GEM entity ID                   object
Parent PermID                          object
Location address                       object
Location address (other language)      object
Municipality                           object
Subnational unit                       object
Country/area                           object
Region                                 object
Coordinates                            object
Coordinate accuracy                    object
GEM wiki page                     

In [75]:
# Check for missing values

# count of NaN per column, sorted so the worst offenders show up first
df.isna().sum().sort_values(ascending=False)


SOE status                            1081
Other plant names (other language)     958
Location address (other language)      794
Owner (other language)                 715
Ferronickel capacity (ttpa)            694
Coking plant capacity (ttpa)           689
Other plant names (English)            551
Pelletizing plant capacity (ttpa)      528
Sinter plant capacity (ttpa)           512
Plant name (other language)            502
Capacity (ttpa)                        358
Met coal source                        115
Plant age                               61
Iron ore source                         16
Parent GEM entity ID                     0
Parent (English)                         0
Owner PermID                             0
Owner                                    0
Owner GEM entity ID                      0
GEM plant ID                             0
Plant name (English)                     0
Parent PermID                            0
Coordinate accuracy                      0
Municipalit

### Question 2: Statistical Summary
**Task:** Generate descriptive statistics for numerical columns.
- What is the average plant capacity? (sum relevant capacity columns in ttpa if needed, e.g. sinter / coking / pelletizing / ferronickel)
- What is the range of latitudes and longitudes? (you will likely need to parse `Coordinates` first — see Part 3 hint)
- What is the distribution of `Plant age (years)`?


In [76]:
# Display descriptive statistics

# Plant age has some "unknown" strings mixed in (seen in Q1), we coerced to numeric
df["Plant age"] = pd.to_numeric(df["Plant age"], errors="coerce")

# Problem ! Coordinates is a single "lat, lon" string, so we then have to split it into two numeric columns
coords = df["Coordinates"].str.split(",", expand=True)
df["Latitude"] = coords[0].astype(float)
df["Longitude"] = coords[1].astype(float)

df[["Capacity (ttpa)", "Plant age", "Latitude", "Longitude"]].describe()


,Capacity (ttpa),Plant age,Latitude,Longitude
count,935.000000,1125.00000,1293.000000,1293.000000
mean,2220.856684,38.78200,30.107078,64.225291
std,2809.649359,36.42716,16.678049,66.403833
min,0.000000,0.00000,-37.831379,-123.163599
25%,700.000000,16.00000,23.504558,27.137563
50%,1200.000000,25.00000,33.962272,87.295932
75%,2700.000000,55.00000,39.976702,115.125838
max,22999.000000,287.00000,67.189096,174.728098


**Answers:**
- Average plant capacity: ~2221 ttpa (only counting the 935 plants that are currently "operating", the rest have no active capacity)
- Latitude range: -37.8 to 67.2 / Longitude range: -123.2 to 174.7 → plants are spread almost across the whole globe
- Plant age: mean ~39 yrs, median 25 yrs → mean > median means a few very old plants (oldest is a 287-yr-old plant in Russia) pull the average up


### Question 3: Geographic Distribution
**Task:** Analyze the geographic distribution of steel plants.
- Which `Country/Area` or `Region` values have the most steel plants?
- What is the distribution of plants by `Owner` (company)?


In [77]:
# Count plants by country/region

print(df["Country/area"].value_counts().head(10))
df["Region"].value_counts()


Country/area
China            458
India            113
United States     90
Iran              56
Japan             42
Russia            31
Türkiye           30
Vietnam           28
Brazil            25
Italy             24
Name: count, dtype: int64


Region
Asia Pacific               765
Europe                     184
North America              113
Middle East                 90
Africa                      51
Eurasia                     47
Central & South America     43
Name: count, dtype: int64

In [78]:
# Count plants by Owner (company)

df["Owner"].value_counts().head(10)


Owner
Nucor Corp                      13
Cleveland-Cliffs Inc            12
Nippon Steel Corp               10
Gerdau Ameristeel Corp           8
Steel Authority of India Ltd     8
Commercial Metals Co             8
SteelAsia Manufacturing Corp     7
United States Steel Corp         6
Liberty Steel Group              6
Tata Steel Ltd                   6
Name: count, dtype: int64

### Question 4: Capacity Analysis
**Task:** Analyze the capacity distribution.
- What is the total global steel production capacity? (sum the capacity columns you are using)
- Which `Owner` values have the highest total capacity?
- How does capacity vary by `Region` or `Country/Area`?

In [79]:
# Calculate total capacity

total_ttpa = df["Capacity (ttpa)"].sum()
print(f"{total_ttpa:,.0f} ttpa -> {total_ttpa/1000:,.0f} Mt/yr")


2,076,501 ttpa -> 2,077 Mt/yr


In [80]:
# Group by Owner and sum capacity

print(df.groupby("Owner")["Capacity (ttpa)"].sum().sort_values(ascending=False).head(10))
print()
# same idea but by region, to see how capacity is spread geographically
print(df.groupby("Region")["Capacity (ttpa)"].sum().sort_values(ascending=False))


Owner
POSCO Holdings Inc              41757.0
Nippon Steel Corp               35395.0
Angang Steel Co Ltd             30250.0
JSW Steel Ltd                   28359.0
Tata Steel Ltd                  26460.0
Hyundai Steel Co                24297.0
Cleveland-Cliffs Inc            23655.0
JFE Steel Corp                  20769.0
Steel Authority of India Ltd    20432.0
Baoshan Iron & Steel Co Ltd     19800.0
Name: Capacity (ttpa), dtype: float64

Region
Asia Pacific               1469080.0
Europe                      207104.0
North America               147087.0
Eurasia                      91774.0
Middle East                  68038.0
Central & South America      55028.0
Africa                       38390.0
Name: Capacity (ttpa), dtype: float64


**Answers:**
- Total global operating capacity: ~2,076,501 ttpa (~2077 Mt/yr)
- Top owner by capacity: POSCO Holdings, then Nippon Steel, Angang Steel, JSW Steel, Tata Steel - different ranking than Q3 (Nucor was #1 by plant count but isn't top 10 by capacity), so most plants ≠ most capacity
- By region: Asia Pacific still dominates, ~71% of world capacity, way ahead of Europe and North America


---
## Part 3: Geospatial Visualization with Plotly

Create interactive maps to visualize the steel plants' locations and characteristics.


### Exercise 1: Basic Scatter Map
**Task:** Create a scatter map showing all steel plant locations.
- Parse `Coordinates` into numeric `Latitude` and `Longitude` (see hint in the code cell)
- Color points by `Country/Area` or `Region`
- Add hover information showing `Plant name (English)`, `Owner`, and capacity


In [81]:
# Create a scatter_geo or scatter_mapbox plot
# Hint: Use plotly.express.scatter_geo() or scatter_mapbox()
#
# Coordinates hint:
#   The dataset usually stores location in a single Coordinates column like "lat, lon".
#   Split it before plotting

# Latitude/Longitude already split out of Coordinates back in Q2, reuse them here

# note: scatter_mapbox got renamed to scatter_map in this plotly version (uses
# MapLibre under the hood, no mapbox token needed), same idea though
fig = px.scatter_map(
    df,
    lat="Latitude",
    lon="Longitude",
    color="Region",
    hover_name="Plant name (English)",
    hover_data=["Owner", "Capacity (ttpa)"],
    title="Steel plants worldwide",
    color_discrete_sequence=px.colors.qualitative.Bold,
    zoom=1,
    height=650,
)

fig.update_traces(marker=dict(size=8, opacity=0.85))
# legend as a semi-transparent overlay on top of the map, map gets full width
fig.update_layout(
    map_style="carto-positron",  # clean light basemap, no clutter
    margin=dict(l=0, r=0, t=60, b=0),
    legend=dict(
        title="Region",
        yanchor="top", y=0.98,
        xanchor="left", x=0.02,
        bgcolor="rgba(255, 255, 255, 0.75)",
        bordercolor="rgba(0, 0, 0, 0.15)",
        borderwidth=1,
    ),
)
fig.show()


### Exercise 2: Sized Markers by Capacity
**Task:** Create a map where marker size represents plant capacity.
- Larger markers for higher capacity plants
- Color by `Owner`
- Include interactive hover details (`Plant name (English)`, `Country/Area`, capacity, etc.)


In [82]:
# Create scatter map with size parameter based on capacity

# size can't handle NaN/negative, so drop plants with no operating capacity
df_cap = df.dropna(subset=["Capacity (ttpa)"]).copy()
df_cap = df_cap[df_cap["Capacity (ttpa)"] > 0]

# 1069 different owners -> way too many colors for a readable legend,
# so keep the top 10 by capacity and lump the rest into "Other"
top_owners = df_cap.groupby("Owner")["Capacity (ttpa)"].sum().nlargest(10).index
df_cap["Owner group"] = df_cap["Owner"].where(df_cap["Owner"].isin(top_owners), "Other")

fig = px.scatter_map(
    df_cap,
    lat="Latitude",
    lon="Longitude",
    size="Capacity (ttpa)",
    color="Owner group",
    hover_name="Plant name (English)",
    hover_data=["Owner", "Country/area", "Capacity (ttpa)"],
    title="Steel plants sized by capacity (top 10 owners highlighted)",
    color_discrete_sequence=px.colors.qualitative.Bold,
    size_max=25,
    zoom=1,
    height=650,
)

fig.update_traces(marker=dict(opacity=0.8))
# legend as a semi-transparent overlay on top of the map instead of a plain
# sidebar, map gets the full width
fig.update_layout(
    map_style="carto-positron",
    margin=dict(l=0, r=0, t=60, b=0),
    legend=dict(
        title="Owner",
        yanchor="top", y=0.98,
        xanchor="left", x=0.02,
        bgcolor="rgba(255, 255, 255, 0.75)",
        bordercolor="rgba(0, 0, 0, 0.15)",
        borderwidth=1,
    ),
)
fig.show()


### Exercise 3: Density Heatmap
**Task:** Create a density map showing concentration of steel plants.
- Use Plotly's density_mapbox to show clustering
- Identify regions with high plant density


In [83]:
# Create density heatmap
# Hint: Use plotly.express.density_mapbox()

# same rename as before: density_mapbox -> density_map in this plotly version
fig = px.density_map(
    df,
    lat="Latitude",
    lon="Longitude",
    radius=15,  # how much each plant "spreads" on the heatmap
    zoom=1,
    height=650,
    title="Steel plant density",
)
fig.update_layout(
    map_style="carto-positron",
    margin=dict(l=0, r=0, t=60, b=0),
)
fig.show()


**Conclusion:** clear hotspot over eastern China, a second (weaker) one over India, and lighter clusters in Western Europe and the US East Coast. Matches the country/region counts from Q3 : plant density basically follows where global steel demand and iron/coal supply chains are.


---
## Part 4: Merging Exposure / Population Data with Assets

Steel plants sit in real places -- next to people, housing, and economic activity. In this part you will attach **granular socio-economic exposure data** to each plant so you can ask: *who and what is near this industrial asset?*

We use **LitPop** (ETH Zurich): a global dataset that combines **population** and **produced capital / asset value** on a fine geographic grid. It is widely used in disaster- and climate-risk analysis as a measure of **exposure**. It is **not** classical environmental monitoring (not air quality, emissions, or weather).

**Goal:** spatially link LitPop grid cells (or sample points) to steel plant locations (nearest neighbor or spatial join), then use the merged fields in maps and later company-level summaries.


### Exercise 1: Load LitPop (Exposure) Data
**Task:** Load the LitPop sample and inspect it.

- **Samples for this lab (recommended):** LitPop sample files are available on Moodle (litpop data). Use these for the merge exercises below.
- **Full LitPop dataset (optional):** [ETH Research Collection – LitPop](https://www.research-collection.ethz.ch/entities/researchdata/12dcfc4f-9d03-463a-8d6b-76c0dc73cdc8)

- Expected columns (may vary by extract): location identifiers, latitude, longitude, population and/or asset-value / exposure fields, etc.

Briefly note what each column represents and the spatial resolution of the sample.


In [84]:
# Load LitPop sample (exposure / population–asset data)

# The Moodle LitPop samples are 3 separate .hdf5 files (one per country: CHN,
# IND, JPN) since those are the only countries we have plant data + LitPop
# data in common for. Each file is a pandas DataFrame saved via to_hdf, load
# them the same way and stack into one table.
import glob
import os

litpop_files = sorted(glob.glob("litpop/LitPop_pc_300_arcsec_*_v1.hdf5"))

frames = []
for f in litpop_files:
    iso3 = os.path.basename(f).split("_")[4]  # e.g. "CHN" out of "..._CHN_v1.hdf5"
    d = pd.read_hdf(f, key="exposures")
    d["country_iso3"] = iso3
    frames.append(d)

litpop = pd.concat(frames, ignore_index=True)
print(litpop.shape)
litpop.head()


(182591, 7)


,value,latitude,longitude,geometry,region_id,impf_,country_iso3
0,5.280440e+09,20.041667,110.208333,POINT (110.20833333 20.04166667),156,1,CHN
1,4.040559e+07,20.041667,110.625000,POINT (110.625 20.04166667),156,1,CHN
2,4.190224e+07,20.041667,110.708333,POINT (110.70833333 20.04166667),156,1,CHN
3,8.813872e+07,19.958333,109.541667,POINT (109.54166667 19.95833333),156,1,CHN
4,1.879947e+08,19.958333,109.625000,POINT (109.625 19.95833333),156,1,CHN


In [85]:
# Inspect LitPop data (columns, dtypes, missing values, value ranges)

print(litpop.dtypes)
print()
print(litpop.isna().sum())
print()
print(litpop["country_iso3"].value_counts())
print()
litpop[["value", "latitude", "longitude"]].describe()


value           float64
latitude        float64
longitude       float64
geometry         object
region_id         int64
impf_             int64
country_iso3     object
dtype: object

value           0
latitude        0
longitude       0
geometry        0
region_id       0
impf_           0
country_iso3    0
dtype: int64

country_iso3
CHN    136991
IND     40101
JPN      5499
Name: count, dtype: int64



,value,latitude,longitude
count,1.825910e+05,182591.000000,182591.000000
mean,3.817856e+08,33.585715,99.540705
std,5.670982e+09,8.816306,17.568848
min,0.000000e+00,6.875000,68.208333
25%,5.023580e+04,27.041667,83.875000
50%,1.295843e+06,33.875000,98.708333
75%,1.401763e+07,40.375000,113.875000
max,5.044057e+11,53.541667,145.791667


**Answers:**
- Columns: `value` = exposed asset value in USD (produced capital, no population in this "pc" extract), `latitude`/`longitude` = grid cell center, `geometry` = same as lat/lon in WKT text, `region_id` = ISO numeric country code, `impf_` = CLIMADA-internal id (not used here)
- Spatial resolution: 300 arc-seconds ≈ 9 km per grid cell at the equator
- 182,591 points total (136,991 CHN / 40,101 IND / 5,499 JPN), no missing values, `value` ranges from 0 to ~504 billion USD (huge range → likely log-scale later)


### Exercise 2: Spatial Join or Nearest Neighbor Matching
**Task:** Attach LitPop exposure attributes to each steel plant based on geographic proximity.
- Match each plant to the **nearest LitPop grid cell / sample point** (or use a spatial join if you work with polygons)
- Consider `geopandas`, a ball-tree / KD-tree nearest-neighbor search, or haversine distances
- Keep plant identifiers and the LitPop fields you will use later (e.g. population, asset value / exposure)

You should end up with one row per plant (or a clear many-to-one rule if you aggregate nearby cells).


In [86]:
# Calculate distances or perform spatial join
# Hint: You might calculate haversine distance or use a spatial library

from sklearn.neighbors import BallTree

# LitPop only covers 3 countries here, so only plants in those countries can
# actually get matched
country_to_iso3 = {"China": "CHN", "India": "IND", "Japan": "JPN"}
df_enriched = df[df["Country/area"].isin(country_to_iso3)].copy()
print(f"{len(df_enriched)} / {len(df)} plants are in a country covered by our LitPop sample")

EARTH_RADIUS_KM = 6371.0

# for each country, build a tree of its LitPop points and query the plants in
# that same country (brute force over all 182k points for every plant would
# be way too slow, a BallTree finds the nearest one in O(log n))
nearest_litpop_idx = pd.Series(index=df_enriched.index, dtype="int64")
nearest_dist_km = pd.Series(index=df_enriched.index, dtype="float64")

for country_name, iso3 in country_to_iso3.items():
    plant_rows = df_enriched[df_enriched["Country/area"] == country_name]
    litpop_rows = litpop[litpop["country_iso3"] == iso3]

    tree = BallTree(np.radians(litpop_rows[["latitude", "longitude"]]), metric="haversine")
    dist, idx = tree.query(np.radians(plant_rows[["Latitude", "Longitude"]]), k=1)

    nearest_litpop_idx.loc[plant_rows.index] = litpop_rows.index[idx.flatten()]
    nearest_dist_km.loc[plant_rows.index] = dist.flatten() * EARTH_RADIUS_KM

df_enriched["litpop_idx"] = nearest_litpop_idx
df_enriched["distance_to_litpop_km"] = nearest_dist_km

df_enriched["distance_to_litpop_km"].describe()


613 / 1293 plants are in a country covered by our LitPop sample


count    613.000000
mean       3.440857
std        1.616343
min        0.031360
25%        2.327039
50%        3.526021
75%        4.383673
max       18.271759
Name: distance_to_litpop_km, dtype: float64

In [87]:
# Merge datasets

# litpop_idx points at a row in `litpop` -> pull its asset value using that index
df_enriched["litpop_asset_value"] = litpop.loc[df_enriched["litpop_idx"], "value"].values
df_enriched = df_enriched.drop(columns=["litpop_idx"])

# nearest-point value is noisy (one 9km cell can be empty right next to a
# dense one), so also sum every LitPop cell within 50km of each plant -> a
# much more stable "how much is exposed around this asset" metric
RADIUS_KM = 50
radius_rad = RADIUS_KM / EARTH_RADIUS_KM
litpop_value_50km = pd.Series(index=df_enriched.index, dtype="float64")

for country_name, iso3 in country_to_iso3.items():
    plant_rows = df_enriched[df_enriched["Country/area"] == country_name]
    litpop_rows = litpop[litpop["country_iso3"] == iso3]

    tree = BallTree(np.radians(litpop_rows[["latitude", "longitude"]]), metric="haversine")
    neighbors = tree.query_radius(np.radians(plant_rows[["Latitude", "Longitude"]]), r=radius_rad)
    values = litpop_rows["value"].to_numpy()
    litpop_value_50km.loc[plant_rows.index] = [values[ix].sum() for ix in neighbors]

df_enriched["litpop_value_50km"] = litpop_value_50km

print(df_enriched.shape)
df_enriched[[
    "Plant name (English)", "Country/area", "Capacity (ttpa)",
    "distance_to_litpop_km", "litpop_asset_value", "litpop_value_50km",
]].head()


(613, 50)


,Plant name (English),Country/area,Capacity (ttpa),distance_to_litpop_km,litpop_asset_value,litpop_value_50km
12,Action Ispat and Power Jharsuguda steel plant,India,375.0,4.629910,6.450750e+08,1.088126e+10
13,Adhunik Metaliks Kuanrmunda steel plant,India,500.0,3.434677,7.211772e+08,1.005962e+10
21,Aichi Steel Chita plant (Tokai),Japan,1495.0,5.250095,1.101869e+11,2.355302e+12
42,Angang Group Xinyang Iron and Steel Co Ltd,China,3600.0,3.392302,3.822503e+08,2.548939e+10
43,Angang Lianzhong Stainless Steel Co Ltd,China,1900.0,3.671572,2.360539e+10,2.328776e+12


**Conclusion:** matched 613/1293 plants (only China, India, Japan are covered by our LitPop sample) to their nearest 300 arc-sec grid cell using a per-country BallTree + haversine distance, avg match distance ~3.4 km, max ~18 km — well within one grid cell's size, so the matches make sense.


### Exercise 3: Visualize Plants with Exposure Context
**Task:** Create a map of steel plants enriched with LitPop fields.
- Color plants by a LitPop metric (e.g. local population or asset exposure)
- Size markers by plant capacity
- Add hover details with both plant attributes (`Plant name (English)`, `Owner`, capacity) and the matched LitPop values

Interpret briefly: where do large plants sit relative to high population / high asset-value areas?


In [88]:
# Create visualization of plants colored by LitPop exposure metrics

# size can't handle NaN, keep only plants with a known operating capacity
plot_df = df_enriched.dropna(subset=["Capacity (ttpa)"])
plot_df = plot_df[plot_df["Capacity (ttpa)"] > 0].copy()

# litpop_value_50km spans several orders of magnitude too, log scale for color
plot_df["log_exposure_50km"] = np.log10(plot_df["litpop_value_50km"])

# both capacity and exposure are heavily skewed (few huge values), so a raw
# Pearson correlation is dominated by outliers -> also check log-log Pearson
# and Spearman (rank-based, doesn't care about the skew at all)
pearson = plot_df["Capacity (ttpa)"].corr(plot_df["litpop_value_50km"])
log_pair = np.log10(plot_df[["Capacity (ttpa)", "litpop_value_50km"]]).dropna()
log_pearson = log_pair.corr().iloc[0, 1]
spearman = plot_df["Capacity (ttpa)"].corr(plot_df["litpop_value_50km"], method="spearman")

print("Correlation between plant capacity and produced capital within 50 km")
print(f"  Pearson (raw)   : {pearson:.3f}")
print(f"  Pearson (log10) : {log_pearson:.3f}")
print(f"  Spearman (rank) : {spearman:.3f}")

fig = px.scatter_map(
    plot_df,
    lat="Latitude",
    lon="Longitude",
    size="Capacity (ttpa)",
    color="log_exposure_50km",
    hover_name="Plant name (English)",
    hover_data=["Owner", "Capacity (ttpa)", "litpop_value_50km"],
    title="Plants sized by capacity, colored by LitPop asset exposure within 50km (log10 USD)",
    color_continuous_scale="Plasma",
    size_max=25,
    # this map only has China/India/Japan data, so auto-center (which defaults
    # to lat=0, lon=0, near Africa) would be wrong -> center on the data instead
    center=dict(lat=plot_df["Latitude"].mean(), lon=plot_df["Longitude"].mean()),
    zoom=2.3,
    height=650,
)
fig.update_traces(marker=dict(opacity=0.85))
fig.update_layout(
    map_style="carto-positron",
    margin=dict(l=0, r=0, t=60, b=0),
    coloraxis_colorbar=dict(title="log10(exposure<br>50km, USD)"),
)
fig.show()

# second view: the actual relationship, log-log scatter colored by country
fig2 = px.scatter(
    plot_df,
    x="litpop_value_50km",
    y="Capacity (ttpa)",
    color="Country/area",
    hover_name="Plant name (English)",
    log_x=True,
    log_y=True,
    opacity=0.7,
    labels={"litpop_value_50km": "Produced capital within 50 km (USD, log)",
            "Capacity (ttpa)": "Operating capacity (ttpa, log)"},
    title="Plant capacity vs surrounding asset value",
)
fig2.update_layout(height=470, margin=dict(l=0, r=0, t=60, b=0))
fig2.show()


Correlation between plant capacity and produced capital within 50 km
  Pearson (raw)   : 0.049
  Pearson (log10) : 0.106
  Spearman (rank) : 0.087


**Interpretation:** using the 50km-radius sum (more robust than a single nearest grid cell), capacity and surrounding asset exposure are weakly positively related / Pearson raw 0.05, log10-log10 0.11, Spearman 0.09. All small but not zero: there's a slight tendency for bigger plants to sit near more economically dense areas, but it's a weak effect, not a real driver. Consistent with the earlier nearest-point result (~0) once we account for how much noisier that single-cell metric was.

Makes sense: steel plants are sited mainly for logistics/raw materials access, not because the surrounding area is economically dense, but it does mean the biggest *disaster-risk* exposure isn't necessarily concentrated at the biggest plants.


---
## Part 5: Company-Level Aggregation

Aggregate data at the company level to analyze corporate footprints — including capacity and the LitPop exposure context you attached in Part 4.


### Exercise 1: Aggregate Metrics by Company
**Task:** Group plants by company (`Owner`) and calculate aggregate metrics.
- Total capacity per company
- Number of plants per company
- Average LitPop exposure metrics per company (from Part 4)
- Geographic spread (e.g. number of `Country/Area` or `Region` values)


In [89]:
# Group by company and aggregate

# general metrics come from the full worldwide `df` (all 1293 plants)
company_agg = df.groupby("Owner").agg(
    total_capacity_ttpa=("Capacity (ttpa)", "sum"),
    n_plants=("GEM plant ID", "count"),
    n_countries=("Country/area", "nunique"),
    n_regions=("Region", "nunique"),
)

# LitPop exposure only exists for plants in China/India/Japan (df_enriched),
# so this column will be NaN for owners with no plant in those 3 countries
avg_litpop = df_enriched.groupby("Owner")["litpop_value_50km"].mean().rename("avg_litpop_value_50km")
company_agg = company_agg.join(avg_litpop)

company_agg = company_agg.sort_values("total_capacity_ttpa", ascending=False)
print(company_agg.shape)
print(f"{company_agg['avg_litpop_value_50km'].notna().sum()} / {len(company_agg)} owners have LitPop coverage")
company_agg.head(10)


(1069, 5)
535 / 1069 owners have LitPop coverage


,total_capacity_ttpa,n_plants,n_countries,n_regions,avg_litpop_value_50km
Owner,,,,,
POSCO Holdings Inc,41757.0,2,1,1,NaN
Nippon Steel Corp,35395.0,10,1,1,1.207883e+12
Angang Steel Co Ltd,30250.0,3,1,1,1.012961e+11
JSW Steel Ltd,28359.0,5,1,1,2.279146e+10
Tata Steel Ltd,26460.0,6,2,1,2.391921e+10
Hyundai Steel Co,24297.0,4,2,2,NaN
Cleveland-Cliffs Inc,23655.0,12,2,1,NaN
JFE Steel Corp,20769.0,5,1,1,3.652170e+12
Steel Authority of India Ltd,20432.0,8,1,1,5.422679e+10


**Answers:** total capacity, plant count, and LitPop exposure per owner are all above (`company_agg`, top 10 by capacity shown). Geographic spread (`n_countries`/`n_regions`) shows most owners are single-country: only **19 / 1069** operate in more than one country. The most spread out is ArcelorMittal (5 countries, 4 regions), followed by Liberty Steel Group (4 countries). Even top-capacity owners like POSCO, Nippon Steel, Angang, JSW are all `n_countries = 1` — being a capacity giant doesn't mean being geographically diversified.


### Exercise 2: Company Headquarters or Centroid
**Task:** Calculate a representative location for each company.
- Option 1: Use the centroid of all plant locations
- Option 2: Use the location of the largest plant
- Option 3: Assign actual headquarters coordinates


In [90]:
# Calculate company representative locations

# going with the centroid (option 1), but weighted by plant capacity rather
# than a plain average: a company's "center of gravity" should be pulled
# toward its biggest plants, not treat a 50 ttpa plant the same as a 20,000 one
def weighted_centroid(group):
    w = group["Capacity (ttpa)"].fillna(0)
    if w.sum() == 0:  # no usable capacity at all -> fall back to plain mean
        return pd.Series({"centroid_lat": group["Latitude"].mean(), "centroid_lon": group["Longitude"].mean()})
    return pd.Series({
        "centroid_lat": np.average(group["Latitude"], weights=w),
        "centroid_lon": np.average(group["Longitude"], weights=w),
    })

centroid = df.groupby("Owner").apply(weighted_centroid, include_groups=False)
company_agg = company_agg.join(centroid)

# also measure how spread out each company's footprint actually is: max
# distance from the centroid to any of its plants (0 for single-plant owners)
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))

tmp = df.merge(company_agg[["centroid_lat", "centroid_lon"]], on="Owner", how="left")
tmp["dist_to_centroid_km"] = haversine_km(tmp["Latitude"], tmp["Longitude"], tmp["centroid_lat"], tmp["centroid_lon"])
company_agg["geographic_spread_km"] = tmp.groupby("Owner")["dist_to_centroid_km"].max()

company_agg[["centroid_lat", "centroid_lon", "geographic_spread_km"]].sort_values("geographic_spread_km", ascending=False).head()


,centroid_lat,centroid_lon,geographic_spread_km
Owner,,,
BlueScope Steel Ltd,-34.463834,150.886191,13766.343189
ArcelorMittal SA,48.992538,3.297453,8420.711943
Liberty Steel Group,41.513249,-87.621438,7806.784309
Wuhan Iron and Steel Co Ltd,30.616222,114.444975,4251.345008
Hyundai Steel Co,36.946278,127.061938,3343.159529


**Note:** capacity-weighted centroid; multi-continent owners (e.g. ArcelorMittal) still land somewhere odd like mid-ocean — `geographic_spread_km` quantifies exactly this (0 for single-plant owners, much higher for spread-out ones).


### Exercise 3: Visualize Company-Level Data
**Task:** Create a map showing companies with aggregated metrics.
- Show one marker per company at the representative location
- Size by total capacity
- Color by average LitPop exposure (or another Part 4 metric)
- Hover information with company summary statistics


In [91]:
# Create company-level visualization

# size can't handle NaN/0, keep companies with known total capacity
map_companies = company_agg[company_agg["total_capacity_ttpa"] > 0].copy()

# 748 companies would clutter the map (mostly tiny single-plant ones), so only
# show companies that either have several plants or a decent total capacity
map_companies = map_companies[
    (map_companies["n_plants"] >= 2) | (map_companies["total_capacity_ttpa"] >= 5000)
]

# same log-scale reasoning as the plant-level map, exposure spans orders of magnitude
map_companies["log_avg_litpop"] = np.log10(map_companies["avg_litpop_value_50km"])

# human-readable hover labels instead of raw numbers
def human_ttpa(v):
    return "n/a" if pd.isna(v) else f"{v/1000:,.1f} Mt/yr"

def human_usd(v):
    if pd.isna(v):
        return "n/a"
    for div, suffix in [(1e12, "T"), (1e9, "B"), (1e6, "M"), (1e3, "k")]:
        if abs(v) >= div:
            return f"${v/div:,.1f}{suffix}"
    return f"${v:,.0f}"

map_companies["capacity_label"] = map_companies["total_capacity_ttpa"].apply(human_ttpa)
map_companies["exposure_label"] = map_companies["avg_litpop_value_50km"].apply(human_usd)
map_companies["spread_label"] = map_companies["geographic_spread_km"].round(0).astype(int).astype(str) + " km"

print(f"{len(map_companies)} / {(company_agg['total_capacity_ttpa'] > 0).sum()} companies shown after anti-clutter filter")

fig = px.scatter_map(
    map_companies,
    lat="centroid_lat",
    lon="centroid_lon",
    size="total_capacity_ttpa",
    color="log_avg_litpop",  # NaN (no LitPop coverage) renders as grey, that's fine
    hover_name=map_companies.index,
    hover_data={
        "n_plants": True, "capacity_label": True, "exposure_label": True,
        "spread_label": True, "n_countries": True,
        "total_capacity_ttpa": False, "log_avg_litpop": False,
    },
    title="Companies: total capacity (size) vs avg. nearby asset exposure (color)",
    color_continuous_scale="Plasma",
    size_max=30,
    zoom=1,
    height=650,
)
fig.update_traces(marker=dict(opacity=0.8))
fig.update_layout(
    map_style="carto-positron",
    margin=dict(l=0, r=0, t=60, b=0),
    coloraxis_colorbar=dict(title="log10(avg exposure<br>50km, USD)"),
)
fig.show()


165 / 748 companies shown after anti-clutter filter


**Conclusion:** anti-clutter filter cuts 748 companies down to 165 shown. Big circles (top owners) cluster in East Asia, matching Q4; grey circles are companies outside LitPop coverage, a real gap, not a data error.


---
## Part 6: Streamlit Dashboard Integration

Prepare your visualizations for deployment in a Streamlit dashboard.


### Exercise 1: Create Dashboard Script Structure
**Task:** Create a Streamlit app file (`app.py`) with the following structure:

```python
# Import streamlit and other necessary libraries

# Set page configuration

# Title and description

# Sidebar for filters
# - Company selector
# - Region/country filter
# - Capacity range slider

# Main content area
# - KPI metrics (total plants, total capacity, etc.)
# - Interactive map
# - Data table

# Footer with data sources and notes
```


### Exercise 1: Prepare Data for Dashboard
**Task:** Save your processed data to files that the dashboard will load.
- Export cleaned plant data
- Export merged plant + LitPop exposure data
- Export company-level aggregations
- Save as CSV or Parquet for efficient loading

In [92]:
# Save processed datasets


### Exercise 2: Display relevant information from your exploratory analysis into the dashboard

In [ ]:
# This cell is for notes/observations about your dashboard
# What works well?
# What could be improved?
# Any performance issues with large datasets?

# What works well:
# - the filters (company/region/country/capacity) combine cleanly since we just
#   chain boolean masks on the same dataframe
# - loading pre-exported CSVs (instead of the raw Excel + LitPop hdf5 + BallTree
#   matching) keeps app.py fast to start and free of the heavy notebook dependencies

# What could be improved:
# - the company selector is a flat multiselect over 1069 names, no search-as-you-type
#   grouping by region would help users find a company faster
# - the dashboard only shows the worldwide plant map, not the LitPop exposure map
#   or the company-level view from Parts 4-5 -> could add tabs for those

# Performance:
# - no issues at this size (1293 plants, 613 with LitPop); px.scatter_map redraws
#   fast even with all filters cleared
# - would probably need to switch to a pre-aggregated/tiled view before this
#   approach scales to tens of thousands of points


---
## Bonus (optional): Deploy to Streamlit Cloud

Deploy your dashboard so it runs in the browser without local setup.

**Task:**
1. Make sure `app.py` and any required data files are in your GitHub repo
2. Go to [https://share.streamlit.io](https://share.streamlit.io) (Streamlit Community Cloud)
3. Sign in with GitHub, select your repo, and deploy `app.py`
4. Copy the public app URL

**Submit:** paste the Streamlit Cloud link in the **Submission info** section at the top **and** include it in your group's submission email (with the GitHub repo URL).


---
## Lab Summary and Key Takeaways

**What you learned:**
- How to perform EDA on geospatial datasets
- Creating interactive maps with Plotly for geospatial data
- Merging LitPop exposure / population data with assets based on geographic proximity
- Aggregating geospatial data at different levels (asset vs. company)
- Building interactive dashboards with Streamlit

**Next Steps:**
- Explore other geospatial libraries (GeoPandas, Folium, Kepler.gl)
- Learn about coordinate reference systems (CRS) and projections
- Practice with other datasets (buildings, utilities, transportation)
- (Bonus) Deploy your dashboard to Streamlit Cloud and share the link in your submission email
